# Building and pruning decision trees with Palmer Penguins



## Libraries and Data



In [ ]:
import numpy as np
import pandas as pd
from plotnine import *
from plotnine.data import penguins
from IPython.display import display

from sklearn.base import clone
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import mean_squared_error, accuracy_score, confusion_matrix
from sklearn.model_selection import (
    GridSearchCV, KFold, StratifiedKFold, train_test_split,
)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import BaggingClassifier, BaggingRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor

RANDOM_STATE = 909

# Preserve the supplied notebook's complete-case sample and predictor choices.
df = penguins.copy().reset_index(drop=True).dropna()

regression_features = [
    "flipper_length_mm", "bill_length_mm", "bill_depth_mm"
]
classification_features = [
    "bill_length_mm", "bill_depth_mm"
]

X_regression = df[regression_features]
y_regression = df["body_mass_g"]
X_classification = df[classification_features]
y_classification = df["species"].astype(str)



## Train Test Split


In [ ]:
X_train_mass, X_test_mass, y_train_mass, y_test_mass = train_test_split(
    X_regression, y_regression,
    test_size=0.20,
    random_state=909
)

X_train_species, X_test_species, y_train_species, y_test_species = train_test_split(
    X_classification, y_classification,
    test_size=0.20,
    random_state=909,
    stratify=y_classification,
)


## Cross-Validation

In [ ]:
reg_cv = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
cls_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

## Bagging

### Bagging Regression: predict body mass

In [ ]:
reg_bagging = BaggingRegressor(
                estimator=DecisionTreeRegressor(max_features=None),
                n_estimators=300,
                max_samples=1.0,
                max_features=1.0,
                bootstrap=True,
                random_state=909,
                n_jobs=-1,
              )

#### Fit Model

In [ ]:
reg_bagging.fit(X_train_mass, y_train_mass)

#### Prediction Error

In [ ]:
y_pred = reg_bagging.predict(X_test_mass)
print("Test MSE:", mean_squared_error(y_test_mass, y_pred))

### Bagging Classification: Penguin Species




In [ ]:
cls_bagging = BaggingClassifier(
                estimator=DecisionTreeClassifier(max_features=None),
                n_estimators=300,
                max_samples=1.0,
                max_features=1.0,
                bootstrap=True,
                random_state=909,
                n_jobs=-1,
            )


#### Fitting Model

In [ ]:
cls_bagging.fit(X_train_species, y_train_species)

#### Classification Error

In [ ]:
y_pred = cls_bagging.predict(X_test_species)
print("Test Error:", accuracy_score(y_test_species, y_pred))

#### Confusion Matrix

In [ ]:
confusion = pd.DataFrame(
    confusion_matrix(
        y_test_species,
        y_pred
    )
)
print(confusion)

## Random Forests

### Random Forests Regressor

In [ ]:
reg_rf = RandomForestRegressor(
           n_estimators=300,
           max_features="sqrt",
           max_depth=None,
           min_samples_leaf=1,
           bootstrap=True,
           random_state=909,
           n_jobs=2
         )

#### Fitting Model

In [ ]:
reg_rf.fit(X_train_mass, y_train_mass)

#### Prediction Error

In [ ]:
y_pred = reg_rf.predict(X_test_mass)
print("Test Error:", mean_squared_error(y_test_mass, y_pred))

### Random Forest Classifier

In [ ]:
cls_rf = RandomForestClassifier(
           n_estimators=300,
           max_features="sqrt",
           max_depth=None,
           min_samples_leaf=1,
           bootstrap=True,
           random_state=909,
           n_jobs=2,
         )

#### Fitting Model

In [ ]:
cls_rf.fit(X_train_species, y_train_species)

#### Classification Error

In [ ]:
y_pred = cls_rf.predict(X_test_species)
print("Test Error:", accuracy_score(y_test_species, y_pred))

#### Confusion Matrix

In [ ]:
confusion = pd.DataFrame(
    confusion_matrix(
        y_test_species,
        y_pred
    )
)
print(confusion)

## Boosting

### Boosting Regressor

In [ ]:
reg_boost = GradientBoostingRegressor(
              loss="squared_error",
              n_estimators=300,
              learning_rate=0.05,
              max_depth=2,
              min_samples_leaf=5,
              subsample=1.0,
              random_state=909
            )

#### Fitting Model

In [ ]:
reg_boost.fit(X_train_mass, y_train_mass)

#### Prediction Error

In [ ]:
y_pred = reg_boost.predict(X_test_mass)
print("Test Error:", mean_squared_error(y_test_mass, y_pred))

### Boosting Classifier

In [ ]:
cls_boost = GradientBoostingClassifier(
              loss="log_loss",
              n_estimators=300,
              learning_rate=0.05,
              max_depth=2,
              min_samples_leaf=5,
              subsample=1.0,
              random_state=42,
            )

#### Fitting Model

In [ ]:
cls_boost.fit(X_train_species, y_train_species)

#### Classification Error

In [ ]:
y_pred = cls_boost.predict(X_test_species)
print("Test Error:", accuracy_score(y_test_species, y_pred))

#### Confusion Matrix

In [ ]:
confusion = pd.DataFrame(
    confusion_matrix(
        y_test_species,
        y_pred
    )
)
print(confusion)